# Part 1g: Custom Dropout & Custom Regularization

**Objective:** Implement custom dropout layers and custom regularizers from scratch in both TF and PyTorch.

---

In [1]:
import numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import torch, torch.nn as nn
import torch.nn.functional as F

(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
X_train, X_test = X_train.astype("float32")/255.0, X_test.astype("float32")/255.0
y_train, y_test = y_train.flatten(), y_test.flatten()
X_train_flat, X_test_flat = X_train.reshape(len(X_train),-1), X_test.reshape(len(X_test),-1)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step


## TensorFlow: Custom Alpha Dropout
Alpha Dropout is designed for SELU activations — it preserves the self-normalizing property by keeping the mean and variance of inputs intact.

In [2]:
# Custom Alpha Dropout for SELU networks (TensorFlow)
class CustomAlphaDropout(layers.Layer):
    """Alpha Dropout: maintains self-normalizing property with SELU."""
    def __init__(self, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.rate = rate

    def call(self, inputs, training=False):
        if not training or self.rate == 0:
            return inputs
        alpha = 1.6732632423543772
        scale = 1.0507009873554805
        alpha_p = -alpha * scale
        # Create dropout mask
        kept = tf.cast(tf.random.uniform(tf.shape(inputs)) >= self.rate, tf.float32)
        # Compute saturation value
        a = ((1 - self.rate) * (1 + self.rate * alpha_p**2))**-0.5
        b = -a * alpha_p * self.rate
        # Apply alpha dropout
        x = inputs * kept + alpha_p * (1 - kept)
        return a * x + b

    def get_config(self):
        config = super().get_config()
        config.update({"rate": self.rate})
        return config

# Custom L1-L2 Regularizer
class CustomElasticRegularizer(keras.regularizers.Regularizer):
    """Custom Elastic Net regularizer with configurable L1/L2 ratio."""
    def __init__(self, l1=0.01, l2=0.01):
        self.l1 = l1
        self.l2 = l2

    def __call__(self, weights):
        return self.l1 * tf.reduce_sum(tf.abs(weights)) + self.l2 * tf.reduce_sum(tf.square(weights))

    def get_config(self):
        return {"l1": float(self.l1), "l2": float(self.l2)}

# Test custom alpha dropout model with SELU
model_custom = keras.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(256, activation='selu', kernel_initializer='lecun_normal',
                 kernel_regularizer=CustomElasticRegularizer(l1=1e-5, l2=1e-4)),
    CustomAlphaDropout(0.1),
    layers.Dense(128, activation='selu', kernel_initializer='lecun_normal',
                 kernel_regularizer=CustomElasticRegularizer(l1=1e-5, l2=1e-4)),
    CustomAlphaDropout(0.1),
    layers.Dense(10, activation='softmax')
])

model_custom.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_custom.summary()
h = model_custom.fit(X_train_flat, y_train, epochs=20, batch_size=256, validation_split=0.2, verbose=1)
print(f"Best val acc: {max(h.history['val_accuracy']):.4f}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │       786,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ custom_alpha_dropout            │ (None, 256)            │             0 │
│ (CustomAlphaDropout)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ custom_alpha_dropout_1          │ (None, 128)            │             0 │
│ (CustomAlphaDropout)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 820,874 (3.13 MB)

 Trainable params: 820,874 (3.13 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.2517 - loss: 2.3104 - val_accuracy: 0.3338 - val_loss: 2.0728
Epoch 2/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3344 - loss: 2.0032 - val_accuracy: 0.3568 - val_loss: 1.9534
Epoch 3/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3647 - loss: 1.9052 - val_accuracy: 0.3759 - val_loss: 1.9069
Epoch 4/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3881 - loss: 1.8408 - val_accuracy: 0.4198 - val_loss: 1.7828
Epoch 5/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.4055 - loss: 1.7892 - val_accuracy: 0.4021 - val_loss: 1.8438
Epoch 6/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.4209 - loss: 1.7520 - val_accuracy: 0.4341 - val_loss: 1.7530
Epoch 7/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4322 - loss: 1.7186 - val_accuracy: 0.4263 - val_loss: 1.7447
Epoch 8/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4363 - loss: 1.7079 - val_accuracy: 0

## TensorFlow: Custom Spatial Dropout
Drops entire feature maps instead of individual neurons — useful for CNNs where adjacent pixels are correlated.

In [3]:
# Custom Concrete Dropout (learned dropout rate)
class ConcreteDropout(layers.Layer):
    """Dropout where the rate is learned during training."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.log_rate = self.add_weight(name='log_rate', shape=(), initializer=tf.initializers.Constant(-2.0), trainable=True)

    def call(self, inputs, training=False):
        if not training:
            return inputs
        rate = tf.sigmoid(self.log_rate)
        mask = tf.cast(tf.random.uniform(tf.shape(inputs)) > rate, tf.float32)
        return inputs * mask / (1.0 - rate)

model_concrete = keras.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(256, activation='relu'), ConcreteDropout(),
    layers.Dense(128, activation='relu'), ConcreteDropout(),
    layers.Dense(10, activation='softmax')
])
model_concrete.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h2 = model_concrete.fit(X_train_flat, y_train, epochs=20, batch_size=256, validation_split=0.2, verbose=1)

# Print learned dropout rates
for layer in model_concrete.layers:
    if isinstance(layer, ConcreteDropout):
        rate = tf.sigmoid(layer.log_rate).numpy()
        print(f"Learned dropout rate: {rate:.4f}")

Epoch 1/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.2682 - loss: 2.0061 - val_accuracy: 0.3355 - val_loss: 1.8425
Epoch 2/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3439 - loss: 1.8191 - val_accuracy: 0.3643 - val_loss: 1.7643
Epoch 3/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3765 - loss: 1.7449 - val_accuracy: 0.3991 - val_loss: 1.6931
Epoch 4/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3884 - loss: 1.7104 - val_accuracy: 0.4018 - val_loss: 1.6911
Epoch 5/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3995 - loss: 1.6846 - val_accuracy: 0.4055 - val_loss: 1.6559
Epoch 6/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4134 - loss: 1.6460 - val_accuracy: 0.4270 - val_loss: 1.6096
Epoch 7/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4219 - loss: 1.6245 - val_accuracy: 0.4340 - val_loss: 1.6020
Epoch 8/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.4272 - loss: 1.6015 - val_accuracy: 0

## PyTorch: Custom Dropout & Regularization

In [4]:
# PyTorch: Custom Gaussian Dropout (multiplicative noise)
class GaussianDropout(nn.Module):
    """Instead of zeroing, multiply by Gaussian noise N(1, rate/(1-rate))."""
    def __init__(self, rate=0.1):
        super().__init__()
        self.rate = rate

    def forward(self, x):
        if self.training and self.rate > 0:
            std = (self.rate / (1.0 - self.rate)) ** 0.5
            noise = torch.randn_like(x) * std + 1.0
            return x * noise
        return x

# PyTorch: Custom L1 Regularization via forward hook
class L1RegularizedLinear(nn.Module):
    """Linear layer with built-in L1 regularization."""
    def __init__(self, in_f, out_f, l1_lambda=1e-5):
        super().__init__()
        self.linear = nn.Linear(in_f, out_f)
        self.l1_lambda = l1_lambda
        self.reg_loss = 0.0

    def forward(self, x):
        self.reg_loss = self.l1_lambda * self.linear.weight.abs().sum()
        return self.linear(x)

class CustomRegNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = L1RegularizedLinear(3072, 256, l1_lambda=1e-5)
        self.drop1 = GaussianDropout(0.1)
        self.fc2 = L1RegularizedLinear(256, 128, l1_lambda=1e-5)
        self.drop2 = GaussianDropout(0.1)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x)); x = self.drop1(x)
        x = F.relu(self.fc2(x)); x = self.drop2(x)
        return self.fc3(x)

    def get_reg_loss(self):
        return self.fc1.reg_loss + self.fc2.reg_loss

# Train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomRegNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.FloatTensor(X_train_flat), torch.LongTensor(y_train)),
    batch_size=256, shuffle=True)

for ep in range(20):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(model(xb), yb) + model.get_reg_loss()
        loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    xt = torch.FloatTensor(X_test_flat).to(device)
    acc = (model(xt).argmax(1) == torch.LongTensor(y_test).to(device)).float().mean()
print(f"PyTorch Custom Model Test Acc: {acc:.4f}")

PyTorch Custom Model Test Acc: 0.5008


## Key Takeaways
- Custom dropout variants: Alpha Dropout (SELU), Gaussian Dropout (multiplicative noise), Concrete Dropout (learned rate)
- Custom regularizers can combine L1/L2 or add domain-specific penalties
- In TF: subclass `layers.Layer` for custom dropout, `keras.regularizers.Regularizer` for custom regularization
- In PyTorch: subclass `nn.Module` for custom layers, add reg loss manually in training loop